In [ ]:
# ==============================================================================
# PARALLEL FACT: RENTAL
# ==============================================================================
from notebooks.helpers import (
    IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger,
    safe_count, generate_batch_id,
)
from notebooks.helpers.silver_transforms import transform_rental_fact
import pandas as pd

logger = setup_logger("parallel_fact_rental")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

rental_batch_id = get_latest_batch_id(spark, "rental")
if not rental_batch_id:
    raise ValueError("No bronze batch_id found for rental; run bronze load first.")
logger.info(f"Using bronze batch_id for rental: {rental_batch_id}")

rental_config = TableConfig(
    table_name="rental",
    business_key="rental_id",
    surrogate_key="rental_key",
    watermark_column="rental_date",
    scd_type=1,
    gold_table_name="fact_rental",
    silver_transform=transform_rental_fact,
    dependencies=["staff", "inventory", "payment"],
)

print("Row Counts (Before):")
print(f"fact_rental: {safe_count(spark, 'fact_rental')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))


In [ ]:
results = pipeline.load_tables([rental_config], force_full=False, bronze_batch_id=rental_batch_id)

display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"fact_rental: {safe_count(spark, 'fact_rental')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
